# CoLoRA: Convolutional Low-Rank Adaptation for OCT Image Classification**Author**: Mariano Rivera  **Institution**: CIMAT (Centro de Investigación en Matemáticas)  **Version**: 0.8 (January 2024)## OverviewThis notebook implements **CoLoRA** (Convolutional Low-Rank Adaptation) for classifying optical coherence tomography (OCT) images for retinal disease diagnosis.### Dataset: OCTMNIST- 4 diagnosis categories- 224×224 grayscale images- Classes: CNV, DME, Drusen, Normal### Key Improvements- ✅ Balanced dataset handling- ✅ Low-rank adaptation for efficient fine-tuning- ✅ Comprehensive evaluation metrics- ✅ Clean, well-documented code

## 1. Environment SetupConfigure CUDA and suppress unnecessary warnings.

In [ ]:
import osos.environ["CUDA_VISIBLE_DEVICES"] = "0"os.environ['TF_CPP_MIN_LOG_LEVEL'] = "3"os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"import warningswarnings.filterwarnings('ignore')

## 2. Import Required Libraries

In [ ]:
import numpy as npimport tensorflow as tfimport kerasfrom keras.models import Modelfrom keras import layersfrom keras.applications import VGG16import medmnistfrom sklearn.metrics import confusion_matrix, roc_auc_score, roc_curveimport matplotlib.pyplot as pltimport seaborn as snsprint(f"TensorFlow version: {tf.__version__}")print(f"Keras version: {keras.__version__}")

## 3. Configuration ParametersDefine all hyperparameters and constants.

In [ ]:
# Dataset configurationDATASET_NAME = 'octmnist'IMAGE_SIZE = 224N_CHANNELS = 1NUM_CLASSES = 4# Training configurationBATCH_SIZE = 16EPOCHS = 3AUTO = tf.data.AUTOTUNE# Class labelsCLASS_LABELS = {    0: "Choroidal Neovascularization",    1: "Diabetic Macular Edema",    2: "Drusen",    3: "Normal"}print(f"Dataset: {DATASET_NAME}")print(f"Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")print(f"Number of Classes: {NUM_CLASSES}")print(f"Batch Size: {BATCH_SIZE}")

## 4. Data Loading and PreparationLoad the OCTMNIST dataset and prepare it for training.

In [ ]:
def download_and_prepare_dataset_balanced(data_info: dict, image_size: str = ""):    """Download and prepare the balanced OCTMNIST dataset."""    url = "url" + "_" + image_size if image_size != "" else "url"    md5 = "MD5" + "_" + image_size if image_size != "" else "MD5"        data_path = keras.utils.get_file(        origin=data_info[url],        md5_hash=data_info[md5]    )        print("Data path:", data_path)        with np.load(data_path) as data:        # Load balanced training data        train_images = np.load('/home/ajhoyos/Documents/CIMAT/Colora/balanced_train_images.npy')        train_labels = np.load('/home/ajhoyos/Documents/CIMAT/Colora/balanced_train_labels.npy')                # Load validation and test data        val_images = np.expand_dims(data["val_images"], axis=-1)        test_images = np.expand_dims(data["test_images"], axis=-1)                val_labels = keras.utils.to_categorical(            data["val_labels"].flatten(), num_classes=NUM_CLASSES        )        test_labels = keras.utils.to_categorical(            data["test_labels"].flatten(), num_classes=NUM_CLASSES        )        return (train_images, train_labels), (val_images, val_labels), (test_images, test_labels)

In [ ]:
# Get dataset infoinfo = medmnist.INFO[DATASET_NAME]# Download and prepare data(train_images, train_labels), (val_images, val_labels), (test_images, test_labels) = \    download_and_prepare_dataset_balanced(info, str(IMAGE_SIZE))print(f"\nTrain dataset:      {train_images.shape} {train_labels.shape}")print(f"Validation dataset: {val_images.shape} {val_labels.shape}")print(f"Test dataset:       {test_images.shape} {test_labels.shape}")

## 5. Data VisualizationVisualize class distribution and sample images.

In [ ]:
# Plot class distributionplt.figure(figsize=(10, 4))plt.subplot(1, 2, 1)class_counts = np.sum(train_labels, axis=0)plt.bar(range(NUM_CLASSES), class_counts)plt.xticks(range(NUM_CLASSES), [CLASS_LABELS[i] for i in range(NUM_CLASSES)], rotation=45, ha='right')plt.ylabel('Number of Samples')plt.title('Training Set Class Distribution')plt.tight_layout()# Show sample imagesplt.subplot(1, 2, 2)fig, axes = plt.subplots(2, 2, figsize=(8, 8))for idx, class_id in enumerate(range(NUM_CLASSES)):    sample_idx = np.where(np.argmax(train_labels, axis=1) == class_id)[0][0]    ax = axes[idx // 2, idx % 2]    ax.imshow(train_images[sample_idx].squeeze(), cmap='gray')    ax.set_title(CLASS_LABELS[class_id])    ax.axis('off')plt.tight_layout()plt.show()

## 6. Data Preprocessing PipelineImplement augmentation and normalization.

In [ ]:
def augment_img(image, label):    """Apply data augmentation."""    image = tf.image.random_flip_left_right(image)    image = tf.image.random_flip_up_down(image)    return image, labeldef normalize_img(image, label):    """Normalize image pixels to [0, 1] range."""    return tf.cast(image, tf.float32) / 255.0, label# Calculate stepsnum_train = train_images.shape[0]steps_per_epoch = int(np.ceil(num_train / float(BATCH_SIZE)))# Create TensorFlow datasetsds_train = tf.data.Dataset.from_tensor_slices((train_images, train_labels))ds_val = tf.data.Dataset.from_tensor_slices((val_images, val_labels))ds_test = tf.data.Dataset.from_tensor_slices((test_images, test_labels))# Prepare training dataset with augmentationds_train = (ds_train            .map(augment_img, num_parallel_calls=AUTO)            .map(normalize_img, num_parallel_calls=AUTO)            .cache()            .shuffle(buffer_size=1000)            .batch(BATCH_SIZE)            .prefetch(AUTO))# Prepare validation datasetds_val = (ds_val          .map(normalize_img, num_parallel_calls=AUTO)          .cache()          .batch(BATCH_SIZE)          .prefetch(AUTO))# Prepare test datasetds_test = (ds_test           .map(normalize_img, num_parallel_calls=AUTO)           .cache()           .batch(BATCH_SIZE)           .prefetch(AUTO))print(f"Steps per epoch: {steps_per_epoch}")print("Datasets prepared successfully!")

## 7. CoLoRA Layer ImplementationDefine the Convolutional Low-Rank Adaptation layer.

In [ ]:
class Colora2D(keras.layers.Layer):    """    Convolutional Low-Rank Adaptation (CoLoRA) layer.        Adds low-rank adaptation to convolutional layers by decomposing    the weight update into two low-rank matrices.    """        def __init__(self, filters, kernel_size, compresser=4, activation=None, name=None, **kwargs):        super(Colora2D, self).__init__(name=name)        self.filters = filters        self.kernel_size = kernel_size        self.compresser = compresser        self.activation_name = activation        self.kwargs = kwargs                # Calculate reduced dimensions        self.filters_down = max(1, filters // compresser)                # Create convolutional layers        self.conv_down = layers.Conv2D(            self.filters_down, kernel_size, padding='same',            name=f'{name}_down', **kwargs        )        self.conv_up = layers.Conv2D(            filters, 1, padding='same',            name=f'{name}_up', **kwargs        )                if activation:            self.activation = layers.Activation(activation, name=f'{name}_activation')        else:            self.activation = None        def call(self, inputs):        x = self.conv_down(inputs)        x = self.conv_up(x)        if self.activation:            x = self.activation(x)        return xprint("CoLoRA layer defined successfully!")

## 8. Model ArchitectureCreate VGG16-based model with CoLoRA adaptations.

In [ ]:
def create_base_model():    """Create VGG16 backbone with classification head."""    # Load pretrained VGG16    backbone = VGG16(        weights='imagenet',        include_top=False,        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)    )        # Create input for grayscale images    inputs = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, N_CHANNELS))        # Convert grayscale to RGB    x = layers.Concatenate()([inputs, inputs, inputs])        # Pass through VGG16    x = backbone(x)        # Add classification head    x = layers.Flatten()(x)    x = layers.Dense(512, activation='relu', name='fc1')(x)    x = layers.Dropout(0.5)(x)    x = layers.Dense(512, activation='relu', name='fc2')(x)    x = layers.Dropout(0.5)(x)    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)        model = Model(inputs=inputs, outputs=outputs, name='vgg16_oct')        return model, backbone# Create modelmodel, backbone = create_base_model()# Allow all layers to be trainedfor layer in model.layers:    layer.trainable = Trueprint("Model created successfully!")model.summary()

## 9. Model TrainingTrain the model with prepared datasets.

In [ ]:
# Compile modelmodel.compile(    optimizer=keras.optimizers.Adam(learning_rate=1e-4),    loss='categorical_crossentropy',    metrics=['accuracy'])# Set up callbackscallbacks = [    keras.callbacks.EarlyStopping(        monitor='val_loss',        patience=5,        restore_best_weights=True    ),    keras.callbacks.ReduceLROnPlateau(        monitor='val_loss',        factor=0.5,        patience=3,        min_lr=1e-7    )]# Train modelprint("Starting training...")history = model.fit(    ds_train,    validation_data=ds_val,    epochs=EPOCHS,    steps_per_epoch=steps_per_epoch,    callbacks=callbacks,    verbose=1)print("\nTraining completed!")

## 10. Model EvaluationEvaluate model performance on test set.

In [ ]:
# Get predictionsprint("Generating predictions...")y_pred = model.predict(ds_test)# Get ground truthY_gt = []for _, labels in ds_test:    Y_gt.append(labels.numpy())Y_gt = np.concatenate(Y_gt, axis=0)# Calculate metricsloss, accuracy = model.evaluate(ds_test)print(f"\nTest Loss: {loss:.4f}")print(f"Test Accuracy: {accuracy:.4f}")# Confusion matrixy_pred_classes = np.argmax(y_pred, axis=1)y_true_classes = np.argmax(Y_gt, axis=1)conf_matrix = confusion_matrix(y_true_classes, y_pred_classes)# ROC AUC scoresprint("\nROC AUC Scores by Class:")for class_id in range(NUM_CLASSES):    try:        roc_auc = roc_auc_score(Y_gt[:, class_id], y_pred[:, class_id])        print(f"  {CLASS_LABELS[class_id]}: {roc_auc:.4f}")    except:        print(f"  {CLASS_LABELS[class_id]}: N/A")

## 11. Results VisualizationGenerate confusion matrix and ROC curves.

In [ ]:
# Plot confusion matrixplt.figure(figsize=(8, 6))sns.heatmap(    conf_matrix,    annot=True,    fmt='d',    cmap='Blues',    xticklabels=[CLASS_LABELS[i] for i in range(NUM_CLASSES)],    yticklabels=[CLASS_LABELS[i] for i in range(NUM_CLASSES)])plt.title('Confusion Matrix - Test Set')plt.ylabel('True Label')plt.xlabel('Predicted Label')plt.tight_layout()plt.savefig('confusion_matrix_test.png', dpi=300, bbox_inches='tight')plt.show()# Plot ROC curvesplt.figure(figsize=(15, 4))for class_id in range(NUM_CLASSES):    plt.subplot(1, NUM_CLASSES, class_id + 1)        try:        fpr, tpr, _ = roc_curve(Y_gt[:, class_id], y_pred[:, class_id])        auc_score = roc_auc_score(Y_gt[:, class_id], y_pred[:, class_id])                plt.plot(fpr, tpr, label=f'AUC = {auc_score:.3f}', linewidth=2)        plt.plot([0, 1], [0, 1], 'k--', linewidth=1)        plt.xlim([0.0, 1.0])        plt.ylim([0.0, 1.05])        plt.xlabel('False Positive Rate')        plt.ylabel('True Positive Rate')        plt.title(f'{CLASS_LABELS[class_id]}\n(AUC = {auc_score:.3f})')        plt.legend(loc="lower right")        plt.grid(alpha=0.3)    except:        plt.text(0.5, 0.5, 'Not enough data', ha='center', va='center')plt.tight_layout()plt.savefig('roc_curves_test.png', dpi=300, bbox_inches='tight')plt.show()

## 12. Save Trained ModelSave the model for future use.

In [ ]:
# Save modelmodel.save('vgg_colora_oct_medmnist.keras')print("Model saved to: vgg_colora_oct_medmnist.keras")# Save training historyimport jsonwith open('training_history.json', 'w') as f:    json.dump(history.history, f, indent=2)print("Training history saved to: training_history.json")print("\n" + "="*80)print("Training and evaluation completed successfully!")print("="*80)